# protocol

> Handle Claude Code's stream-json protocol, including NDJSON transport, control requests, and deferred tool results

In [ ]:
#| default_exp protocol

`fastclaude.protocol` connects Python applications to a running Claude Code process without the Agent SDK. `ClaudeProto` exchanges JSON messages with the process and yields its conversation events. The `ClaudeRun` runner in `fastclaude.core` starts and stops the process.

The application executes the tools it supplies to fastclaude. When Claude requests one of these tools, fastclaude ends the turn before execution. The application runs the tool and supplies its result on the next run. `ClaudeProto` returns that result when Claude resumes the pending call. It never executes the tool itself.

In [ ]:
#| export
import asyncio, json, os
from fastcore.utils import *
from fastcore.funccall import get_schema

In [ ]:
from fastcore.test import *
from fastclaude.session import ant_data, sess_dir
from collections import Counter
import shutil, sys, tempfile

## Conversation events

Claude Code writes newline-delimited JSON to standard output with `--output-format stream-json --verbose --include-partial-messages`. Each line contains one message. The runner uses `--input-format stream-json` to send JSON messages through standard input.

The examples use a saved recording of Claude's built-in Bash tool. Unlike the application tools discussed below, Claude Code executes this tool itself. The recording ships with the package. `mk_stream_fixture` returns its path without starting Claude when the file exists. To record a new run, delete `stream_path` before calling it. Recording starts Claude in a temporary directory and uses the installed CLI's credentials.

In [ ]:
stream_path = ant_data.parent/'stream.jsonl'

In [ ]:
async def mk_stream_fixture(path=None):  # chkstyle: ignore-node
    "Capture one live claude run's raw stream-json stdout as the checked-in stream fixture"
    path = Path(path) if path else stream_path
    if path.exists(): return path
    prompt = 'think hard: first work out 17*23-4 in your head, then use the Bash tool to run: echo flux-41.7 . Then reply with exactly the tool output followed by your arithmetic answer.'
    argv = ['claude','-p','--output-format','stream-json','--verbose','--include-partial-messages','--model','sonnet','--allowedTools','Bash']
    td = tempfile.mkdtemp()
    p = await asyncio.create_subprocess_exec(*argv, stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE, limit=2**25, cwd=td)
    out,_ = await p.communicate(prompt.encode())
    assert not p.returncode, f'claude exited {p.returncode}'
    path.write_bytes(out)
    shutil.rmtree(sess_dir(td), ignore_errors=True)
    return path

In [ ]:
await mk_stream_fixture()

Path('/Users/jhoward/aai-ws/fastclaude/fastclaude/data/stream.jsonl')

The recording contains conversation messages, generation updates, and process status events. Count each type before selecting the events to consume.

In [ ]:
evs = dict2obj(stream_path.read_jsonl())
len(evs),Counter(e.type for e in evs)

(26,
 Counter({'stream_event': 16,
          'system': 5,
          'assistant': 2,
          'rate_limit_event': 1,
          'user': 1,
          'result': 1}))

`stream_event` messages wrap Anthropic's streaming events, such as `message_start` and `content_block_delta`. These include incremental text and tool arguments. The `assistant` events contain completed content blocks. Tool results appear in `user` events.

The completed events have the same `uuid` and `message` content as their records in the session transcript, described in `fastclaude.session`. The protocol layer returns these events unchanged.

Use the incremental events to display progress. Use the completed events to assemble the conversation without duplicating content. In this recording, concatenating the argument updates reproduces the completed tool call's input. The tool result identifies that call through `tool_use_id`.

In [ ]:
tu_ev = first(e for e in evs if e.type=='assistant' and e.message.content[0].type=='tool_use')
tr_ev = first(e for e in evs if e.type=='user')
test_eq(tr_ev.message.content[0].tool_use_id, tu_ev.message.content[0].id)
deltas = [d.event.delta for d in evs if d.type=='stream_event' and d.event.type=='content_block_delta']
test_eq(json.loads(''.join(d.partial_json for d in deltas if d.type=='input_json_delta')), obj2dict(tu_ev.message.content[0].input))
tu_ev.message.content[0].name, tr_ev.message.content[0].content

('Bash', 'flux-41.7')

The recording also contains `system` and `rate_limit_event` messages. The `system` events include session initialization and status changes. They also include `hook_started` and `hook_response` from the user's configured hooks, which run even without an interactive terminal. Other runs can include `thinking_tokens` system events, or `command_lifecycle` and `attachment` records on resume.

Applications must tolerate event types they do not recognize, including new types added by later CLI versions. `ClaudeProto.events` forwards every message except the control messages it handles. It does not filter conversation events by a list of known types.

## Reading JSON lines

`read_msgs` uses `StreamReader.readline` to read a complete line before decoding it. Pipe reads do not need to align with JSON messages.

Set the stream's size limit when starting the subprocess. A message containing an image can exceed asyncio's default limit. The fastclaude runner uses `limit=2**25`, allowing lines up to 32 MiB.

In [ ]:
#| export
async def read_msgs(
    stream, # An asyncio `StreamReader` of NDJSON, e.g. a claude process's stdout
):
    "Yield decoded JSON messages from `stream`"
    while line := await stream.readline():
        if not (s := line.strip()): continue
        try: yield json.loads(s)
        except json.JSONDecodeError as e:
            if line.endswith(b'\n'): raise ValueError(f'bad NDJSON line: {s[:200]!r}') from e
            return  # Ignore an invalid final fragment without a newline.

This subprocess writes the recording in 37-byte chunks. `read_msgs` must return the same messages regardless of where those writes divide the data.

In [ ]:
emit = f"""import sys
data = open({str(stream_path)!r}, 'rb').read()
for i in range(0, len(data), 37):
    sys.stdout.buffer.write(data[i:i+37])
    sys.stdout.buffer.flush()"""
p = await asyncio.create_subprocess_exec(sys.executable, '-c', emit, stdout=asyncio.subprocess.PIPE, limit=2**25)
got = [m async for m in read_msgs(p.stdout)]
await p.wait()
test_eq(got, list(obj2dict(evs)))
test_eq(got[-1]['type'], 'result')

`read_msgs` skips blank lines. It accepts valid JSON on the final line without a trailing newline. Invalid JSON followed by a newline raises `ValueError`. Invalid JSON at the end of the stream without a newline ends iteration without yielding that fragment.

In [ ]:
def feedr(b):
    "A `StreamReader` pre-fed with `b`, at EOF"
    r = asyncio.StreamReader()
    r.feed_data(b)
    r.feed_eof()
    return r

test_eq([m async for m in read_msgs(feedr(b'{"a": 1}\n\n{"b": 2}'))], [dict(a=1), dict(b=2)])
test_eq([m async for m in read_msgs(feedr(b'{"a": 1}\n{"cut": tru'))], [dict(a=1)])
bad = read_msgs(feedr(b'not json\n'))
with expect_fail(ValueError, contains='bad NDJSON'): await bad.__anext__()

## Describing tools


Pass `mk_tools` a list of functions, schema dictionaries, or both. For a function, `tool_spec` uses its signature and docstring to generate a schema. It returns an existing schema dictionary unchanged.

Each schema contains `name`, `description`, and `inputSchema`. The example combines the annotated `flux_meter` function with a dictionary describing a Python tool.


In [ ]:
#| export
def tool_spec(
    t, # An annotated callable, or a schema dict with `name`, `description`, `inputSchema`
):
    "Return a schema dictionary for a function or an existing schema"
    return get_schema(t, pname='inputSchema') if callable(t) else t

def mk_tools(
    tools, # Tools in either `tool_spec` form
):
    "Return the tool definitions as schema dictionaries"
    return [tool_spec(t) for t in listify(tools)]

In [ ]:
async def flux_meter(unit:str='kf') -> str:
    "Read the flux."
    return f'flux: 41.7 {unit}'

py_schema = dict(name='py', description='Run code in the kernel',
    inputSchema=dict(type='object', properties=dict(code=dict(type='string')), required=['code']))
schemas = mk_tools([flux_meter, py_schema])
test_eq([s['name'] for s in schemas], ['flux_meter','py'])
schemas[0]

{'name': 'flux_meter',
 'description': 'Read the flux.\n\nReturns:\n- type: string',
 'inputSchema': {'type': 'object',
  'properties': {'unit': {'description': '',
    'default': 'kf',
    'type': 'string'}}}}

## Returning a tool result

A resumed session can contain a tool call whose result the application has already computed. The runner passes this result to `ClaudeProto` as `held`. Claude requests the pending tool again on resume. The protocol responds with `held` rather than running the tool again.

Claude Code expects MCP content blocks in this response. Anthropic image blocks store their media type and data inside `source`. MCP image blocks put `mimeType` and `data` directly on the block. `_mcp_content` converts images to the MCP format and returns other blocks unchanged.

In [ ]:
#| export
def _mcp_content(block):
    "Translate one Anthropic content block to MCP content"
    if block.get('type')!='image': return block
    src = block['source']
    return dict(type='image', data=src['data'], mimeType=src['media_type'])

In [ ]:
text_block = dict(type='text', text='flux: 41.7 gauss')
image_block = dict(type='image', source=dict(type='base64', media_type='image/png', data='aGVsbG8='))
test_eq(_mcp_content(text_block), text_block)
converted_image = _mcp_content(image_block)
test_eq(converted_image, dict(type='image', mimeType='image/png', data='aGVsbG8='))
converted_image

{'type': 'image', 'data': 'aGVsbG8=', 'mimeType': 'image/png'}

`tool_reply` builds the response from content blocks or a string. It converts a string to a text block and converts images with `_mcp_content`. Set `is_error=True` to report a tool failure.

In [ ]:
#| export
def tool_reply(content, is_error=False):
    "Convert Anthropic content to an MCP tool result"
    if isinstance(content, str): content = [dict(type='text', text=content)]
    return dict(content=[_mcp_content(b) for b in content], isError=is_error)

In [ ]:
reply = tool_reply([text_block, image_block], is_error=True)
test_eq(reply['content'], [text_block, converted_image])
test_eq(reply['isError'], True)
reply

{'content': [{'type': 'text', 'text': 'flux: 41.7 gauss'},
  {'type': 'image', 'data': 'aGVsbG8=', 'mimeType': 'image/png'}],
 'isError': True}

`jrpc` builds JSON-RPC messages with version `2.0`. It includes `params` only when arguments are present. An `id` of `None` omits the ID field, making the message a notification.

In [ ]:
#| export
def jrpc(
    method, # JSON-RPC method name
    id=None, # Request id; None makes a notification
    **params, # The call's `params`, omitted when empty
):
    "Build a JSON-RPC request or notification"
    r = dict(jsonrpc='2.0', id=id, method=method)
    if id is None: r.pop('id')
    if params: r['params'] = params
    return r

In [ ]:
test_eq(jrpc('ping', 2), dict(jsonrpc='2.0', id=2, method='ping'))
jrpc('tools/call', 9, name='flux_meter', arguments=dict(unit='gauss'))

{'jsonrpc': '2.0',
 'id': 9,
 'method': 'tools/call',
 'params': {'name': 'flux_meter', 'arguments': {'unit': 'gauss'}}}

## Answering MCP requests

Claude Code uses MCP requests to discover fastclaude's tools and collect results. For these tools, it sends requests inside its control messages rather than through a separate MCP connection.

`mcp_dispatch` handles `initialize`, `notifications/*`, `ping`, `tools/list`, and `tools/call`. It returns JSON-RPC responses without sending them. It does not implement the rest of MCP or require the `mcp` package.


In [ ]:
#| export
def mcp_dispatch(
    msg, # One JSON-RPC message from the CLI
    schemas, # Tool schemas to advertise, from `mk_tools`
    held=None, # The `tool_reply` for the call Claude re-invokes on resume, if any
    server='fastclaude', # Server name reported to the CLI
):
    "The JSON-RPC response for `msg`, or None for a notification"
    m,i = msg.get('method'), msg.get('id')
    def res(r): return dict(jsonrpc='2.0', id=i, result=r)
    if m == 'initialize':
        pv = nested_idx(msg, 'params', 'protocolVersion') or '2024-11-05'
        return res(dict(protocolVersion=pv, capabilities=dict(tools={}), serverInfo=dict(name=server, version='1.0')))
    if m and m.startswith('notifications/'): return None
    if m == 'ping': return res({})
    if m == 'tools/list': return res(dict(tools=schemas))
    if m == 'tools/call':
        if held: return res(held)
        return res(dict(content=[dict(type='text', text=f"no held result for {nested_idx(msg, 'params', 'name')}")], isError=True))
    return dict(jsonrpc='2.0', id=i, error=dict(code=-32601, message=f'method not found: {m}'))

The `disp` helper supplies the example's schemas to `mcp_dispatch`. Initialization echoes the requested protocol version. If the request omits a version, the dispatcher uses `2024-11-05`. A notification returns `None`. A tool listing returns the supplied schemas.


In [ ]:
schemas = mk_tools([flux_meter, py_schema])
def disp(method, id=None, held=None, **p): return mcp_dispatch(jrpc(method, id, **p), schemas, held)

r = disp('initialize', 1, protocolVersion='2025-06-18')
test_eq(r['result']['protocolVersion'], '2025-06-18')
test_eq(disp('notifications/initialized'), None)
test_eq(disp('tools/list', 3)['result']['tools'], schemas)
[m['name'] for m in schemas]


['flux_meter', 'py']

For `tools/call`, the dispatcher returns the supplied `held` result. It does not match the requested name or arguments against that result. The runner must provide the result for the session's pending call.


In [ ]:
gauss = tool_reply('flux: 41.7 gauss')
flux_result = disp('tools/call', 9, gauss, name='flux_meter', arguments=dict(unit='gauss'))['result']
test_is(flux_result, gauss)
flux_result

{'content': [{'type': 'text', 'text': 'flux: 41.7 gauss'}], 'isError': False}

A tool call without a stored result returns `isError=True` and a message naming the requested tool. This indicates a disagreement between the permission hook and the result handler. The hook should defer a tool call when no result exists.

An unsupported method returns JSON-RPC error `-32601`, not a tool result. Both error responses let Claude finish the request rather than waiting for an answer.


In [ ]:
r = disp('tools/call', 10, name='flux_meter', arguments={})['result']
test_eq((r['isError'], r['content'][0]['text']), (True, 'no held result for flux_meter'))
test_eq(disp('resources/list', 4)['error']['code'], -32601)
r


{'content': [{'type': 'text', 'text': 'no held result for flux_meter'}],
 'isError': True}

## Control requests and responses

Claude Code and `ClaudeProto` use `control_request` messages to request operations from each other. Each request has a `request_id`. A `control_response` repeats that ID to identify the request it answers.

`ctrl_req` builds a request. `ctrl_ok` and `ctrl_err` build success and error responses.

In [ ]:
#| export
def ctrl_req(rid, **req):
    "Build a `control_request` message"
    return dict(type='control_request', request_id=rid, request=req)

def ctrl_ok(rid, **resp):
    "Build a successful `control_response` message"
    return dict(type='control_response', response=dict(subtype='success', request_id=rid, response=resp))

def ctrl_err(rid, error):
    "Build an error `control_response` message"
    return dict(type='control_response', response=dict(subtype='error', request_id=rid, error=str(error)))

Create one `ClaudeProto` for each subprocess. The `server` name must match the runner's MCP configuration. Pass the tool definitions through `tools` and any previously computed result through `held`.

`initialize` starts the handshake. If tools are present, it registers a `PreToolUse` hook for names matching `mcp__<server>__.*`. Claude Code calls this hook before using those tools. With a result in `held`, the hook returns `allow`. Without one, it returns `defer`. Deferral ends the turn before tool execution.

`send_req` sends a control request and waits for a response with the same ID. An error response raises `RuntimeError`. The `timeout` limits how long it waits. `interrupt` sends Claude Code's interrupt request to end the current turn without stopping the process.

Run the `events` iterator while awaiting these methods. The iterator reads the responses that complete their pending requests.


In [ ]:
#| export
class ClaudeProto:
    "Exchange control messages and conversation events with one Claude Code process"
    def __init__(self,
        proc, # An asyncio subprocess speaking stream-json on piped stdin/stdout
        tools=None, # Tool schemas to advertise, in either `tool_spec` form
        held=None, # The `tool_reply` Claude collects on resume; any other call defers
        server='fastclaude', # SDK MCP server name, matching the `--mcp-config` entry
    ):
        self.proc,self.held,self.server = proc,held,server
        self.schemas = mk_tools(tools or [])
        self._lock,self._n,self._pending,self._inflight = asyncio.Lock(),0,{},{}

    async def send(self, obj):
        "Write one JSON message to Claude Code's standard input"
        async with self._lock:
            self.proc.stdin.write(json.dumps(obj, ensure_ascii=False).encode()+b'\n')
            await self.proc.stdin.drain()

    async def send_req(self, req, timeout=60):
        "Send a control request and return its response dictionary"
        self._n += 1
        rid = f'req_{self._n}_{os.urandom(4).hex()}'
        fut = asyncio.get_running_loop().create_future()
        self._pending[rid] = fut
        await self.send(ctrl_req(rid, **req))
        try: return await asyncio.wait_for(fut, timeout)
        finally: self._pending.pop(rid, None)

    async def initialize(self, timeout=120):
        "Start the handshake and register the tool permission hook"
        hooks = dict(PreToolUse=[dict(matcher=f'mcp__{self.server}__.*', hookCallbackIds=['defer'])]) if self.schemas else None
        return await self.send_req(dict(subtype='initialize', hooks=hooks), timeout)

    async def interrupt(self, timeout=30):
        "Ask Claude Code to end the current turn without stopping the process"
        return await self.send_req(dict(subtype='interrupt'), timeout)

In [ ]:
#| export
@patch
def _resolve(self:ClaudeProto, msg):
    "Resolve the pending request identified by a `control_response`"
    r = msg.get('response') or {}
    if (fut := self._pending.get(r.get('request_id'))) and not fut.done():
        if r.get('subtype')=='error': fut.set_exception(RuntimeError(r.get('error') or 'control request failed'))
        else: fut.set_result(r.get('response') or {})

@patch
def _route(self:ClaudeProto, req):
    "Allow tool use when `held` contains a result, otherwise defer"
    return dict(hookSpecificOutput=dict(hookEventName='PreToolUse', permissionDecision='allow' if self.held else 'defer'))

Incoming `mcp_message` requests contain JSON-RPC messages for `mcp_dispatch`. Incoming `hook_callback` requests ask for the `PreToolUse` decision. Other request subtypes receive error responses.

After answering `tools/call`, `_mcp` clears `held`. The next hook callback then returns `defer`. For a JSON-RPC notification, `_mcp` returns an empty result inside the control response. The control request still requires an answer even though the notification itself does not.

In [ ]:
#| export
@patch
def _mcp(self:ClaudeProto, msg):
    "Answer an MCP request and clear `held` after a tool call"
    resp = mcp_dispatch(msg, self.schemas, self.held, self.server)
    if msg.get('method')=='tools/call': self.held = None
    return dict(mcp_response=resp or dict(jsonrpc='2.0', result={}))

@patch
async def _handle(self:ClaudeProto, rid, req):
    "Answer a control request from Claude Code"
    try:
        st = req.get('subtype')
        if st=='mcp_message': resp = self._mcp(req.get('message') or {})
        elif st=='hook_callback': resp = self._route(req)
        else: raise ValueError(f"unsupported control request: {st}")
        await self.send(ctrl_ok(rid, **resp))
    except asyncio.CancelledError: raise
    except Exception as e: await self.send(ctrl_err(rid, e))

`events` handles each incoming control request in a separate asyncio task. It matches control responses to pending requests. A `control_cancel_request` cancels the matching handler without sending a response. Cancellation for an unknown ID has no effect.

The iterator yields all other messages unchanged. When iteration exits, `aclose` cancels the request handlers and pending response futures. It does not stop the subprocess.

In [ ]:
#| export
@patch
async def events(self:ClaudeProto):
    "Handle control messages and yield other events from Claude Code"
    try:
        async for m in read_msgs(self.proc.stdout):
            t = m.get('type')
            if t=='control_response': self._resolve(m)
            elif t=='control_request':
                rid = m.get('request_id')
                task = asyncio.create_task(self._handle(rid, m.get('request') or {}))
                self._inflight[rid] = task
                task.add_done_callback(lambda _,rid=rid: self._inflight.pop(rid, None))
            elif t=='control_cancel_request':
                if task := self._inflight.pop(m.get('request_id'), None): task.cancel()
            else: yield m
    finally: await self.aclose()



In [ ]:
#| export
@patch
async def aclose(self:ClaudeProto):
    "Cancel in-flight handlers and pending requests"
    for t in list(self._inflight.values()): t.cancel()
    for f in self._pending.values():
        if not f.done(): f.cancel()

The following test replaces Claude Code with in-memory streams. `feed` supplies incoming messages through a `StreamReader`. `_Sink` records the messages that `ClaudeProto` writes back.

The test starts with `held` containing the result `flux: 41.7 kf`. This represents a session resuming after the application has run `flux_meter`. No tool or model runs during this test.


In [ ]:
class _Sink:
    "Record messages written to a simulated process"
    def __init__(self): self.msgs = []
    def write(self, b): self.msgs.append(json.loads(b))
    async def drain(self): pass

rdr = asyncio.StreamReader()
peer = AttrDict(stdout=rdr, stdin=_Sink())
proto = ClaudeProto(peer, tools=[flux_meter], held=tool_reply('flux: 41.7 kf'))
def feed(o): rdr.feed_data(json.dumps(o).encode()+b'\n')



Start `events` in a background task before sending control requests. The `out` list collects the conversation events it yields. The `hook` helper constructs Claude Code's permission requests for `flux_meter`.

In [ ]:
out = []
async def drain():
    async for m in proto.events(): out.append(m)
t = asyncio.create_task(drain())
def hook(rid, **inp): feed(ctrl_req(rid, subtype='hook_callback', callback_id='defer', input=dict(tool_name='mcp__fastclaude__flux_meter', tool_input=inp)))


Request `h1` asks permission to run `flux_meter`. The stored result lets `ClaudeProto` answer `allow`. Request `m1` then asks for the tool result through MCP.

In [ ]:
hook('h1', unit='kf')
await asyncio.sleep(0.05)
feed(ctrl_req('m1', subtype='mcp_message', server_name='fastclaude', message=jrpc('tools/call', 1, name='flux_meter', arguments=dict(unit='kf'))))
await asyncio.sleep(0.05)
[m['response']['request_id'] for m in peer.stdin.msgs]

['h1', 'm1']

Request `h2` asks permission for another tool call after the first has consumed `held`. This time `ClaudeProto` must answer `defer`. The test also sends a cancellation for an unknown ID, followed by a conversation result and end of input.

In [ ]:
hook('h2', unit='gauss')
feed(dict(type='control_cancel_request', request_id='nobody'))
await asyncio.sleep(0.05)
feed(dict(type='result', subtype='success'))
rdr.feed_eof()
await t
test_eq(proto.held, None)

Check the responses by request ID. The MCP response must contain the original `kf` result. The cancellation must produce no response. The conversation output must contain the `result` event and none of the control messages.

In [ ]:
resp = {m['response']['request_id']: m['response']['response'] for m in peer.stdin.msgs}
test_eq(resp['h1']['hookSpecificOutput']['permissionDecision'], 'allow')
test_eq(resp['m1']['mcp_response']['result']['content'][0]['text'], 'flux: 41.7 kf')
test_eq(resp['h2']['hookSpecificOutput']['permissionDecision'], 'defer')
test_eq('nobody' in resp, False)
test_eq([m['type'] for m in out], ['result'])
resp['h2']


{'hookSpecificOutput': {'hookEventName': 'PreToolUse',
  'permissionDecision': 'defer'}}

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()